# MUSA 650 Homework 2: Supervised Land Use Classification with Google Earth Engine

## 1. Setup

In [99]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import folium
import shapely
from shapely.geometry import Point, Polygon, mapping, shape
import geemap
import ee
import os
import csv
import ast
import json
import time
import glob
import random
import requests
from collections import Counter
from rasterio.crs import CRS
from rasterio.features import rasterize
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score

In [2]:
ee.Authenticate()

True

In [3]:
ee.Initialize()

In [4]:
shapefile_path = "C:/Users/MaJia/OneDrive - PennO365/文档/GitHub/musa-650-spring-2025/assignments/data/tam.shp" 
roi = gpd.read_file(shapefile_path)
roi = roi.to_crs(epsg=4326)

roi_center = roi.geometry.centroid.iloc[0]
map = folium.Map(location=[roi_center.y, roi_center.x], zoom_start=7, tiles="cartodb positron")

folium.GeoJson(roi, name="Region of Interest").add_to(map)

map

C:\Users\MaJia\AppData\Local\Temp\ipykernel_11120\229568009.py:5: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  roi_center = roi.geometry.centroid.iloc[0]


For this study, I have chosen Tamaulipas, Mexico as the region of interest (ROI) due to the variety of land cover types present. This area contains developed areas, barrier islands, wetlands, lagoons and vegetation zones, making it an ideal site for building a training dataset for supervised classification. The area’s geomorphology and ecology are also rather special and offers a good opportunity to examine environmental changes, coastal processes and human impacts over time. With the help of Landsat 8 satellite imagery, I hope to build a model that help us correctly identify land cover types within the area, supporting further research efforts.

## 2. Data Collection and Feature Engineering

### 2.1 Collecting and Labeling Training Data

### 2.2 Feature Engineering

In [22]:
roi_coords = [list(poly.exterior.coords) for poly in roi.geometry.iloc[0].geoms]
roi_ee = ee.Geometry.MultiPolygon([roi_coords])
buffer_radius = 2500

landcover_class = {
    #"urban": "50",   
    #"bare": "60",    
    #"water": "80",   
    "vegetation": "10",  
}

num_points = 1000
output_dir = "C:/Users/MaJia/OneDrive - PennO365/文档/GitHub/musa-650-spring-2025/assignments/training_data_2_HW2/"
output_filename = "training_dataset_4.csv"
output_path = os.path.join(output_dir, output_filename)

worldcover = ee.ImageCollection("ESA/WorldCover/v100").first().select("Map")

def generate_random_points(roi, num_points):
    seed = int(time.time())
    return ee.FeatureCollection.randomPoints(roi, num_points, seed)

def buffer_feature(feature):
    return feature.buffer(buffer_radius)

def check_land_cover(feature, target_class):
    landcover_stats = worldcover.reduceRegion(
        reducer=ee.Reducer.frequencyHistogram(),  
        geometry=feature.geometry(),
        scale=10,
        bestEffort=True
    ).get("Map")
    
    target_fraction = ee.Dictionary(landcover_stats).get(target_class, 0)
    total = ee.Dictionary(landcover_stats).values().reduce(ee.Reducer.sum())
    match = ee.Algorithms.If(ee.Number(target_fraction).divide(total).gte(0.7), 1, 0)
    return feature.set("match", match)

all_features = ee.FeatureCollection([])

for label, target_class in landcover_class.items():
    print(f"Processing class: {label}")
    
    total_matched_count = 0
    matched_features_list = []
    max_attempts = 150
    attempts = 0

    while total_matched_count < 100 and attempts < max_attempts:
        attempts += 1
        print(f"Attempt {attempts}: Generating new points for {label}")
        random_points = generate_random_points(roi_ee, num_points)
        buffered_points = random_points.map(buffer_feature)
        
        print("Start searching...")
        matched_points = buffered_points.map(lambda f: check_land_cover(f, target_class)) \
                                        .filter(ee.Filter.eq("match", 1))
        matched_count = matched_points.size().getInfo()
        print(f"New matched points for {label}: {matched_count}")
        
        if matched_count > 0:
            matched_list = matched_points.toList(matched_count)
            for i in range(matched_count):
                feature = ee.Feature(matched_list.get(i))
                feature = feature.set("land_cover", label)
                feature = feature.setGeometry(feature.geometry().centroid())
                matched_features_list.append(feature)
        
        total_matched_count = len(matched_features_list)
        print(f"Total accumulated matched points for {label}: {total_matched_count}")
        if total_matched_count >= 100:
            break
    
    if len(matched_features_list) > 0:
         matched_features_fc = ee.FeatureCollection(matched_features_list)
         all_features = all_features.merge(matched_features_fc)

Processing class: vegetation
Attempt 1: Generating new points for vegetation
Start searching...
New matched points for vegetation: 137
Total accumulated matched points for vegetation: 137


In [50]:
landsat = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2") \
    .filterBounds(roi_ee) \
    .filterDate("2023-01-01", "2023-12-31") \
    .filter(ee.Filter.lt("CLOUD_COVER", 20)) \
    .median()

ndvi = landsat.normalizedDifference(["SR_B5", "SR_B4"]).rename("NDVI")
ndbi = landsat.normalizedDifference(["SR_B6", "SR_B5"]).rename("NDBI")
mndwi = landsat.normalizedDifference(["SR_B3", "SR_B6"]).rename("MNDWI")

dem = ee.Image("USGS/SRTMGL1_003").rename("elevation")
slope = ee.Terrain.slope(dem).rename("slope")

#ndvi_norm = ndvi.unitScale(0, 1).rename("NDVI_norm")
#ndbi_norm = ndbi.unitScale(0, 1).rename("NDBI_norm")
#mndwi_norm = mndwi.unitScale(0, 1).rename("MNDWI_norm")
#elev_norm = dem.unitScale(0, 1).rename("elevation_norm")
#slope_norm = slope.unitScale(0, 1).rename("slope_norm")

composite = ndvi_norm.addBands(ndbi) \
                    .addBands(mndwi) \
                    .addBands(dem) \
                    .addBands(slope)

training_data = composite.sampleRegions(
    collection=all_features,
    properties=["land_cover"],
    scale=30,  
    geometries=True
)

download_url = training_data.getDownloadURL(filetype='CSV')
print("Download URL:", download_url)

response = requests.get(download_url)
if response.status_code == 200:
    with open(output_path, 'wb') as f:
        f.write(response.content)
    print("Dataset saved locally at:", output_path)
else:
    print("Error downloading file:", response.status_code)

Download URL: https://earthengine.googleapis.com/v1/projects/674377568019/tables/4c3fcbc06e1ff0d9bb7e619d6670e705-46e1f775e5c47b7ab3047a665604d861:getFeatures
Dataset saved locally at: C:/Users/MaJia/OneDrive - PennO365/文档/GitHub/musa-650-spring-2025/assignments/training_data_2_HW2/training_dataset_4.csv


## 3. Model Training and Evaluation

### 3.1 Model Training

In [83]:
input_dir = r"C:\Users\MaJia\OneDrive - PennO365\文档\GitHub\musa-650-spring-2025\assignments\training_data_2_HW2"
files = glob.glob(os.path.join(input_dir, "*.csv"))

td_list = []
for file in files:
    td = pd.read_csv(file)
    td_list.append(td)
    
all_data = pd.concat(td_list, ignore_index=True)

features_to_scale = ["MNDWI", "NDBI", "NDVI_norm", "elevation", "slope"]
scaler = MinMaxScaler()
all_data[features_to_scale] = scaler.fit_transform(all_data[features_to_scale])

all_data.rename(columns={"NDVI_norm": "NDVI"}, inplace=True)

features_to_scale = ["MNDWI", "NDBI", "NDVI", "elevation", "slope"]

mapping = {"urban": 0, "bare": 1, "water": 2, "vegetation": 3}
all_data["land_cover"] = all_data["land_cover"].str.lower().map(mapping)

X = all_data[features_to_scale]
y = all_data["land_cover"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

results = []

current_features = features_to_scale.copy()
best_accuracy = 0
best_features = current_features.copy()

iteration = 0
while len(current_features) > 1:
    iteration += 1
    print(f"\nIteration {iteration}, using features: {current_features}")
    
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_train[current_features], y_train)
    y_pred = rf.predict(X_val[current_features])
    accuracy = accuracy_score(y_val, y_pred)
    print("Validation Accuracy:", accuracy)
    
    results.append({'features': current_features.copy(), 'accuracy': accuracy})
    if accuracy > best_accuracy:
         best_accuracy = accuracy
         best_features = current_features.copy()
    
    importances = rf.feature_importances_
    importance_df = pd.DataFrame({
         'Feature': current_features,
         'Importance': importances
    }).sort_values(by='Importance', ascending=True)

    least_important = importance_df.iloc[0]['Feature']
    print(importance_df)
    
    current_features.remove(least_important)

print("\nSummary of iterations:")
for res in results:
    print(f"Features: {res['features']}, Accuracy: {res['accuracy']}")

print("\nBest combination of features:", best_features)
print("Highest validation accuracy:", best_accuracy)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train[features_to_scale], y_train)
y_pred = rf.predict(X_val[features_to_scale])


Iteration 1, using features: ['MNDWI', 'NDBI', 'NDVI', 'elevation', 'slope']
Validation Accuracy: 0.9624060150375939
     Feature  Importance
1       NDBI    0.096061
4      slope    0.132589
0      MNDWI    0.186306
2       NDVI    0.255946
3  elevation    0.329099

Iteration 2, using features: ['MNDWI', 'NDVI', 'elevation', 'slope']
Validation Accuracy: 0.9473684210526315
     Feature  Importance
3      slope    0.083325
0      MNDWI    0.236571
1       NDVI    0.318360
2  elevation    0.361744

Iteration 3, using features: ['MNDWI', 'NDVI', 'elevation']
Validation Accuracy: 0.9473684210526315
     Feature  Importance
1       NDVI    0.323166
0      MNDWI    0.329207
2  elevation    0.347627

Iteration 4, using features: ['MNDWI', 'elevation']
Validation Accuracy: 0.9473684210526315
     Feature  Importance
0      MNDWI    0.474329
1  elevation    0.525671

Summary of iterations:
Features: ['MNDWI', 'NDBI', 'NDVI', 'elevation', 'slope'], Accuracy: 0.9624060150375939
Features: ['MNDW

When assessed by variable importance score, **elevation, NDVI, and MNDWI** appeared to be the top 3 most influential features in the model. While some indicators have lower importance scores, tests with different combinations of features show that keeping all features lead to highest validation accuracy.

### 3.2 Accuracy Assessment

In [78]:
def create_grid(r, s):
    b = r.bounds().getInfo()['coordinates'][0]
    xs = [i[0] for i in b]
    ys = [i[1] for i in b]
    mnx, mxx = min(xs), max(xs)
    mny, mxy = min(ys), max(ys)
    d = s / 111000.0
    fs = []
    x = mnx
    while x < mxx:
        y = mny
        while y < mxy:
            fs.append(ee.Feature(ee.Geometry.Rectangle([x, y, x + d, y + d])))
            y += d
        x += d
    return ee.FeatureCollection(fs)

roi_ee2 = ee.Geometry.Polygon([[
        [-97.8, 25.5],
        [-97.8, 24.0],
        [-97.3, 24.0],
        [-97.3, 25.5],
    ]])
grid_fc = create_grid(roi_ee2, 2500)
centroids = grid_fc.map(lambda f: f.centroid())

landsat = (
    ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
    .filterBounds(roi_ee)
    .filterDate("2023-01-01", "2023-12-31")
    .filter(ee.Filter.lt("CLOUD_COVER", 20))
    .median()
)

ndvi = landsat.normalizedDifference(["SR_B5", "SR_B4"]).rename("NDVI")
ndbi = landsat.normalizedDifference(["SR_B6", "SR_B5"]).rename("NDBI")
mndwi = landsat.normalizedDifference(["SR_B3", "SR_B6"]).rename("MNDWI")
dem = ee.Image("USGS/SRTMGL1_003").rename("elevation")
slope = ee.Terrain.slope(dem).rename("slope")
composite2 = ndvi.addBands(ndbi).addBands(mndwi).addBands(dem).addBands(slope)

step = 100
all_pred_data = []
start = 0

while True:
    sub_fc = ee.FeatureCollection(centroids.toList(step, start))
    sub_info = sub_fc.getInfo()
    feats = sub_info.get("features", [])
    if not feats:
        break
    sample_fc = composite2.sampleRegions(collection=sub_fc, scale=30, geometries=True)
    url = sample_fc.getDownloadURL("CSV")
    r = requests.get(url)
    out_file = f"results/prediction_batch_{start}.csv"
    with open(out_file, "wb") as f:
        f.write(r.content)
    df_sub = pd.read_csv(out_file)
    all_pred_data.append(df_sub)
    start += step
    print(start)

pred = pd.concat(all_pred_data, ignore_index=True)
scaler = MinMaxScaler()
pred[["NDVI", "NDBI", "MNDWI", "elevation", "slope"]] = scaler.fit_transform(
    pred[["NDVI", "NDBI", "MNDWI", "elevation", "slope"]]
)

100
200
300
400
500
600
700
800
900
1000
1100
1200
1300
1400
1500
1600


In [90]:
def make_square(pt, cell_size_m=2500):
    if pt is None:
        return None
    half_deg = (cell_size_m / 111000.0) / 2.0
    cx, cy = pt.x, pt.y
    x1, y1 = cx - half_deg, cy - half_deg
    x2, y2 = cx + half_deg, cy + half_deg
    return Polygon([(x1, y1), (x2, y1), (x2, y2), (x1, y2), (x1, y1)])

predictions = rf.predict(pred[["MNDWI", "NDBI", "NDVI", "elevation", "slope"]])
pred["predicted"] = predictions

pred_gdf = gpd.GeoDataFrame(
    pred,
    geometry=pred[".geo"].apply(safe_geometry_load),
    crs="EPSG:4326"
)

pred_gdf["geometry"] = pred_gdf["geometry"].apply(lambda g: make_square(g, cell_size_m=2500))

map2 = folium.Map(location=[24.75, -97.55], zoom_start=9, tiles="cartodb positron")
cdict = {0: "#006400", 1: "#B2B2B2", 2: "#0032C8", 3: "#00A000"}

folium.GeoJson(
    pred_gdf.to_json(),
    style_function=lambda f: {
        "fillColor": cdict.get(f["properties"]["predicted"], "#FFFFFF"),
        "color": "transparent",
        "weight": 0,
        "fillOpacity": 0.5,
    }
).add_to(map2)

map2

Above is a predicted land cover classification map in the barrier islands region, in Tamaulipas. As shown in the map, the model accurately identifies the area with water bodies (as marked by blue markers) and vegetated areas (as marked by green markers) within mainland, while mistakenly recognizes all points outside of mainland as vegetated areas.

In [84]:
cm = confusion_matrix(y_val, y_pred)
acc = accuracy_score(y_val, y_pred)
prec = precision_score(y_val, y_pred, average=None)
rec = recall_score(y_val, y_pred, average=None)

print("Confusion Matrix:\n", cm)
print("Overall Accuracy:", acc)
print("Precision (per class):", prec)
print("Recall (per class):", rec)
print("\nDetailed Classification Report:")
print(classification_report(y_val, y_pred, target_names=["urban","bare","water","vegetation"]))

Confusion Matrix:
 [[27  2  0  1]
 [ 0 30  1  0]
 [ 0  1 30  0]
 [ 0  0  0 41]]
Overall Accuracy: 0.9624060150375939
Precision (per class): [1.         0.90909091 0.96774194 0.97619048]
Recall (per class): [0.9        0.96774194 0.96774194 1.        ]

Detailed Classification Report:
              precision    recall  f1-score   support

       urban       1.00      0.90      0.95        30
        bare       0.91      0.97      0.94        31
       water       0.97      0.97      0.97        31
  vegetation       0.98      1.00      0.99        41

    accuracy                           0.96       133
   macro avg       0.96      0.96      0.96       133
weighted avg       0.96      0.96      0.96       133



According to the confusion matrix, **Urban** and **Bare** are the two most easily mistaken land cover types, probably due to the fact that they have similar material (concrete/asphalt/soil) and similar spectral reflections.

In [88]:
esa_worldcover = ee.Image("ESA/WorldCover/v200/2021").select("Map").clip(roi_ee2)

def add_ee_layer(self, ee_object, vis_params, name):
    if isinstance(ee_object, ee.Image):
        map_id_dict = ee.Image(ee_object).getMapId(vis_params)
        folium.raster_layers.TileLayer(
            tiles=map_id_dict["tile_fetcher"].url_format,
            attr="Map Data © Google Earth Engine",
            name=name,
            overlay=True,
            control=True
        ).add_to(self)
    elif isinstance(ee_object, ee.FeatureCollection):
        map_id_dict = ee_object.getMapId(vis_params)
        folium.features.GeoJson(
            data=map_id_dict["tile_fetcher"].url_format,
            name=name,
            overlay=True,
            control=True
        ).add_to(self)

folium.Map.add_ee_layer = add_ee_layer

esa_vis = {
    "min": 10,
    "max": 100,
    "palette": [
        "#006400", 
        "#FFBB22", 
        "#FFFF4C", 
        "#F096FF", 
        "#FA0000", 
        "#B4B4B4", 
        "#F0F0F0", 
        "#0064C8", 
        "#0096A0", 
        "#00CF75", 
        "#FAE6A0"  
    ]
}

map3 = folium.Map(location=[24.75, -97.55], zoom_start=9, tiles="cartodb positron")
map3.add_ee_layer(esa_worldcover, esa_vis, "ESA WorldCover")
folium.LayerControl().add_to(m)
map3

By visually comparing predicted map with actual land cover classification map, we can see that the main deficit for the model is a lack of account of diverse land cover types. While accurately dipicting the land-water division, the model fails to account diverse types of vegetation - shrubs(orange), grass(yellow) and cropland(pink).

In [103]:
minx, miny, maxx, maxy = pred_gdf.total_bounds

pixel_size_deg = 2500 / 111000.0

width = int(np.ceil((maxx - minx) / pixel_size_deg))
height = int(np.ceil((maxy - miny) / pixel_size_deg))

transform = rasterio.transform.from_origin(
    west=minx,
    north=maxy,
    xsize=pixel_size_deg,
    ysize=pixel_size_deg
)

shapes = ((geom, val) for geom, val in zip(pred_gdf.geometry, pred_gdf["predicted"]))

rasterized = rasterio.features.rasterize(
    shapes=shapes,
    out_shape=(height, width),
    fill=0,                 
    transform=transform,
    all_touched=True,       
    dtype=rasterio.uint8
)

profile = {
    "driver": "GTiff",
    "height": height,
    "width": width,
    "count": 1,
    "dtype": "uint8", 
    "transform": transform
}

output_tif = "results/predicted_classification_map.tif"
with rasterio.open(output_tif, "w", **profile) as dst:
    dst.write(rasterized, 1)

print(f"GeoTIFF saved as: {output_tif}")

GeoTIFF saved as: results/predicted_classification_map.tif


In [108]:
report = classification_report(y_val, y_pred, output_dict=True)

results_csv = "results/classification_metrics.csv"
with open(results_csv, "w", newline="") as f:
    writer = csv.writer(f)
    
    writer.writerow(["Overall Accuracy", acc])
    
    writer.writerow([])
    
    writer.writerow(["Class", "Precision", "Recall", "F1-score", "Support"])
    for cls, metrics in report.items():
        if cls.isdigit():
            writer.writerow([
                cls,
                metrics["precision"],
                metrics["recall"],
                metrics["f1-score"],
                metrics["support"]
            ])
    
    writer.writerow([])
    
    writer.writerow(["Confusion Matrix"])
    num_classes = cm.shape[0]
    writer.writerow([""] + [f"Pred_{i}" for i in range(num_classes)])
    for i, row in enumerate(cm):
        writer.writerow([f"True_{i}"] + list(row))

print(f"Classification metrics saved to: {results_csv}")

Classification metrics saved to: results/classification_metrics.csv


## 4. Reflection Questions

**What limitations did you run into when completing this assignment? What might you do differently if you repeated it, or what might you change if you had more time and/or resources?**

The main limitation I encountered is Google Earth Engine's incapability in dealing with large requests. When gathering the dataset, I split my download content into 4 batches (<100 entries each time). A similar problem appeared when handling prediction dataset. In the end, I decided to 1) reduce prediction area; 2) sample the centroids from each 2500m*2500m region and 3) download by batches to solve the problem.

One thing I might do differently is to include more land cover types into the training dataset - to better represent the diverse geographical characteristics of Tamaulipas.

**What was the impact of feature engineering? Which layers most contributed to the model? Did you expect this? Why or why not?**

In this assignment, feature engineering transformed spectral values into meaningful indicators, which helped us in understanding their impacts. 

In all engineered features, elevation contributed the most to the model, followed by MNDWI and NDVI. This makes sense because elevation is the most impactful indicator in definitive decisions - water bodies wouldn't be raised from ground level, while vegetated areas are usually higher up. MNDWI and NDVI are also impactful contributing to the model due to their ability in identifying water and vegetated areas.

**Did you find it difficult to create the training data by hand? Did you notice any issues with class imbalance? If so, how might you resolve this in the future (hint: consider a different sampling technique).**

I didn't create the training data by hand; instead, I create a loop that iteratively generate 1,000 random points within the ROI. During each loop, the program will generate a buffer area of 2,500 meters with these generated points as centroids, and decide whether 70% of land cover type within that area is compatible with desired type (by looking at ESA WorldCover dataset). At the end of each loop, the program will keep the geometries with matched land cover type, and continue if matched sample did not reach 100. When doing this, searching for urban area and bare area were incredibly slower, due to these areas being significantly smaller in size. If required dataset is really large, we could consider use a pre-defined urban area as urban-specific searching area.

**Did your model perform better on one class than another? Why? Can you think of a reason that this might be good or bad depending on the context?**

Actually, the testing set were too small to tell accuracy difference between the classes. The general accuracy for each class is all quite high, not excluding the possibility of overfitting.